In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

### Load datasets

In [2]:
customers_df = pd.read_csv("Customers.csv")
products_df = pd.read_csv("Products.csv")
transactions_df = pd.read_csv("Transactions.csv")

### Lookalike Model

In [4]:
# Merge datasets
merged_df = transactions_df.merge(customers_df, on='CustomerID', how='left')
merged_df = merged_df.merge(products_df, on='ProductID', how='left')

In [5]:
# Feature Engineering
# Aggregating transactional data per customer
customer_features = merged_df.groupby('CustomerID').agg({
    'TotalValue': ['sum', 'mean'],
    'Quantity': 'sum',
    'TransactionID': 'nunique',
    'Category': lambda x: x.nunique(),
    'TransactionDate': lambda x: (pd.to_datetime(x).max() - pd.to_datetime(x).min()).days
}).reset_index()

In [6]:
# Renaming columns
customer_features.columns = [
    'CustomerID', 'TotalSpent', 'AvgSpent', 'TotalQuantity', 'TransactionCount',
    'UniqueCategories', 'RecencyDays'
]

In [8]:
# Adding customer profile features
customer_profile = customers_df[['CustomerID', 'Region']]

In [9]:
# One-hot encode the Region
customer_profile = pd.get_dummies(customer_profile, columns=['Region'])

In [10]:
# Merge features
final_features = customer_features.merge(customer_profile, on='CustomerID', how='left')

In [11]:
# Fill missing values
final_features.fillna(0, inplace=True)

In [12]:
# Scaling features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(final_features.drop('CustomerID', axis=1))

In [13]:
# Similarity Calculation
similarity_matrix = cosine_similarity(scaled_features)

In [14]:
# Mapping CustomerID to index
customer_ids = final_features['CustomerID'].tolist()
customer_index = {id_: index for index, id_ in enumerate(customer_ids)}

In [15]:
# Generating Lookalike Recommendations
lookalike_data = {}

for cust_id in customer_ids[:20]:  # For CustomerID C0001 - C0020
    idx = customer_index[cust_id]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Exclude self (first entry) and get top 3 similar customers
    top_3 = [(customer_ids[i], round(score, 4)) for i, score in sim_scores[1:4]]
    
    lookalike_data[cust_id] = top_3

In [16]:
# Exporting Results
lookalike_df = pd.DataFrame([
    {'CustomerID': cust, 'SimilarCustomerID': sim_cust, 'SimilarityScore': score}
    for cust, lookalikes in lookalike_data.items()
    for sim_cust, score in lookalikes
])

In [18]:
# Exporting to CSV
lookalike_df.to_csv('FirstName_LastName_Lookalike.csv', index=False)